# 🚛 Capacitated Vehicle Routing Problem (CVRP)
## Modélisation MILP & Résolution avec PuLP

---

## 📐 1. Modélisation Mathématique (MILP)

### Données du problème

| Notation | Description |
|----------|-------------|
| $V = \{0, 1, \ldots, n\}$ | Nœuds ($0$ = dépôt, $1..n$ = clients) |
| $K$ | Ensemble des véhicules |
| $d_i$ | Demande du client $i$ |
| $Q$ | Capacité de chaque véhicule |
| $c_{ij}$ | Distance de l'arc $(i,j)$ |

---

### Variables de décision

$$x_{ijk} \in \{0, 1\} \quad \forall i,j \in V,\ k \in K$$

$x_{ijk} = 1$ si le véhicule $k$ emprunte l'arc $(i \to j)$, 0 sinon.

$$u_{ik} \geq 0 \quad \forall i \in V \setminus \{0\},\ k \in K$$

Charge cumulée du véhicule $k$ à l'arrivée au nœud $i$ (variables MTZ).

---

### Fonction objectif

$$\min \sum_{k \in K} \sum_{i \in V} \sum_{j \in V} c_{ij}\, x_{ijk}$$

---

### Contraintes

**(C1) Chaque client visité exactement une fois**

$$\sum_{k \in K} \sum_{j \in V} x_{ijk} = 1 \quad \forall i \in V \setminus \{0\}$$

**(C2) Conservation du flux**

$$\sum_{j} x_{jik} = \sum_{j} x_{ijk} \quad \forall i \in V \setminus \{0\},\ k \in K$$

**(C3) Départ dépôt au plus une fois par véhicule**

$$\sum_{j \in V \setminus \{0\}} x_{0jk} \leq 1 \quad \forall k \in K$$

**(C4) Retour dépôt au plus une fois par véhicule**

$$\sum_{i \in V \setminus \{0\}} x_{i0k} \leq 1 \quad \forall k \in K$$

**(C5) Capacité + élimination sous-tours (MTZ)**

$$u_{jk} \geq u_{ik} + d_j - Q\,(1 - x_{ijk}) \quad \forall i \in V,\ j \in V\setminus\{0\},\ i\neq j,\ k \in K$$

**(C6) Bornes sur la charge cumulée**

$$d_i \leq u_{ik} \leq Q \quad \forall i \in V \setminus \{0\},\ k \in K$$

> **Note MTZ** : Les contraintes (C5)-(C6) de Miller-Tucker-Zemlin éliminent simultanément les sous-tours et garantissent le respect de la capacité.

## 📦 2. Installation des dépendances

In [5]:
!pip install pulp matplotlib numpy --quiet
print('Dependances installees')

Dependances installees


In [1]:
import sys
import os

print("Python utilisé :", sys.executable)
# Cela doit afficher un chemin finissant par .venv\Scripts\python.exe

Python utilisé : c:\doc Arthur\AA Code\M1EDE\.venv\Scripts\python.exe


In [6]:
import sys
!"{sys.executable}" -m pip install pulp --quiet

## 🔧 3. Imports & Configuration

In [7]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pulp, warnings, time
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#0f1117',
    'axes.edgecolor':   '#2d3748',
    'axes.labelcolor':  '#e2e8f0',
    'text.color':       '#e2e8f0',
    'xtick.color':      '#718096',
    'ytick.color':      '#718096',
    'grid.color':       '#1a202c',
    'font.family':      'monospace',
})
print('Imports OK — PuLP:', pulp.__version__)

Imports OK — PuLP: 3.3.0


## 🗺️ 4. Chargement de l'instance JSON

Trois fichiers sont disponibles. Changer `INSTANCE_FILE` pour en sélectionner un.

| Fichier | Clients | Véhicules | Capacité | Difficulté |
|---------|---------|-----------|----------|------------|
| `instance_small.json` | 8 | 2 | 35 | 🟢 Facile |
| `instance_medium.json` | 12 | 3 | 45 | 🟡 Moyen |
| `instance_large.json` | 15 | 4 | 40 | 🔴 Difficile |

> **Import dans Colab** : icône 📁 (panneau gauche) → glisser-déposer les 3 fichiers `.json`.

In [11]:
# ═══════════════════════════════════════════════════
# CHOISIR L'INSTANCE  ← modifier cette ligne
# ═══════════════════════════════════════════════════
INSTANCE_FILE = '../data/instance_small.json'


In [9]:
# @title
# ───────────────────────────────────────────────────
# PARSER JSON
# ───────────────────────────────────────────────────
def parse_instance(filepath):
    """Parse un fichier JSON d'instance CVRP et retourne un dictionnaire normalisé."""
    with open(filepath, 'r') as f:
        raw = json.load(f)

    # -- Champs obligatoires --
    required = ['name', 'parameters', 'depot', 'clients', 'distance_matrix']
    missing  = [k for k in required if k not in raw]
    if missing:
        raise ValueError(f'Champs manquants dans le JSON : {missing}')

    # -- Paramètres --
    params     = raw['parameters']
    n_clients  = int(params['n_clients'])
    n_vehicules = int(params['n_vehicles'])
    capacity   = int(params['capacity'])
    grid_size  = int(params.get('grid_size', 100))

    # -- Dépôt --
    depot_id    = int(raw['depot']['id'])
    depot_coord = list(map(float, raw['depot']['coordinates']))
    if depot_id != 0:
        raise ValueError(f'Le dépôt doit avoir id=0, trouvé id={depot_id}')

    # -- Clients --
    clients_raw = sorted(raw['clients'], key=lambda c: int(c['id']))
    if len(clients_raw) != n_clients:
        raise ValueError(
            f'n_clients={n_clients} mais {len(clients_raw)} clients trouvés')

    client_ids    = [int(c['id']) for c in clients_raw]
    client_coords = [list(map(float, c['coordinates'])) for c in clients_raw]
    client_demands = [float(c['demand']) for c in clients_raw]

    if any(d < 0 for d in client_demands):
        raise ValueError('Demandes negatives detectees')
    if any(d > capacity for d in client_demands):
        raise ValueError('Un client a une demande superieure a la capacity')

    # -- Matrice de distances --
    dm = raw['distance_matrix']
    n  = n_clients + 1
    if len(dm) != n or any(len(row) != n for row in dm):
        raise ValueError(
            f'distance_matrix doit etre ({n}x{n}), '
            f'trouvé ({len(dm)}x{len(dm[0]) if dm else "?"})')
    dist_matrix = [[float(dm[i][j]) for j in range(n)] for i in range(n)]

    # Vérif symétrie (tolérance 1e-3)
    for i in range(n):
        for j in range(n):
            if abs(dist_matrix[i][j] - dist_matrix[j][i]) > 1e-3:
                print(f'  [WARN] Matrice non symétrique en ({i},{j})')
        if dist_matrix[i][i] != 0.0:
            print(f'  [WARN] Diagonale non nulle en ({i},{i})')

    # -- Stats optionnelles --
    stats = raw.get('stats', {})

    return {
        'name':         raw['name'],
        'description':  raw.get('description', ''),
        'n_clients':    n_clients,
        'n_vehicules':   n_vehicules,
        'capacity':     capacity,
        'grid_size':    grid_size,
        'depot_coord':  depot_coord,
        'client_ids':   client_ids,
        'client_coords':client_coords,
        'demands':      [0.0] + client_demands,   # index 0 = dépôt
        'coords':       [depot_coord] + client_coords,
        'dist_matrix':  dist_matrix,
        'stats':        stats,
    }

In [12]:
# ───────────────────────────────────────────────────
# Chargement & extraction des variables de travail
# ───────────────────────────────────────────────────
inst        = parse_instance(INSTANCE_FILE)

N_CLIENTS   = inst['n_clients']
N_vehicules  = inst['n_vehicules']
capacity    = inst['capacity']
grid_size   = inst['grid_size']
n           = N_CLIENTS + 1
coords      = np.array(inst['coords'])
depot_coord = np.array(inst['depot_coord'])
demands     = np.array(inst['demands'])
dist        = np.array(inst['dist_matrix'])

# ───────────────────────────────────────────────────
# Affichage du résumé
# ───────────────────────────────────────────────────
s = inst['stats']
print('=' * 58)
print(f'  Instance  : {inst["name"]}')
print(f'  {inst["description"]}')
print('=' * 58)
print(f'  Noeuds      : {n} (1 dépôt + {N_CLIENTS} clients)')
print(f'  Véhicules   : {N_vehicules}')
print(f'  Capacité    : {capacity}')
print(f'  grid      : {grid_size} x {grid_size}')
if s:
    print(f'  Dem. totale : {s.get("total_demand","?")}  '
          f'moy={s.get("avg_demand","?")}  '
          f'min={s.get("min_demand","?")}  '
          f'max={s.get("max_demand","?")}')
    print(f'  Veh. min th.: {s.get("theoretical_min_vehicules","?")}')
print('-' * 58)
print(f'  {"ID":<5} {"Coordonnées":^22} {"Demande":>8} {"% capacité":>10}')
print(f'  {"──":<5} {"──────────────────────":^22} {"───────":>8} {"──────────":>10}')
print(f'  {0:<5} {str(tuple(depot_coord)):^22} {"— dépôt —":>8}')
for c in inst['client_ids']:
    d   = int(demands[c])
    pct = d / capacity * 100
    bar = '|' * int(pct / 5)
    print(f'  {c:<5} {str(tuple(coords[c])):^22} {d:>8}   {pct:5.1f}%  {bar}')
print('=' * 58)
print('  Extrait matrice de distances (5 premiers noeuds) :')
print(f'  {"":5}' + ''.join(f'{j:8d}' for j in range(min(5,n))))
for i in range(min(5, n)):
    row = ''.join(f'{dist[i][j]:8.1f}' for j in range(min(5,n)))
    print(f'  {i:<5}{row}')
if n > 5: print(f'  ... ({n}x{n} au total)')
print('=' * 58)
print('  Parsing OK')

  Instance  : instance_small
  Petite instance : 8 clients, 2 véhicules, capacité 35. Idéale pour valider le modèle rapidement.
  Noeuds      : 9 (1 dépôt + 8 clients)
  Véhicules   : 2
  Capacité    : 35
  grid      : 80 x 80
  Dem. totale : 58  moy=7.25  min=5  max=10
  Veh. min th.: ?
----------------------------------------------------------
  ID         Coordonnées        Demande % capacité
  ──    ──────────────────────  ─────── ──────────
  0     (np.float64(40.0), np.float64(40.0)) — dépôt —
  1     (np.float64(31.2), np.float64(71.6))        8    22.9%  ||||
  2     (np.float64(56.2), np.float64(46.9))        6    17.1%  |||
  3     (np.float64(15.9), np.float64(15.9))       10    28.6%  |||||
  4     (np.float64(9.1), np.float64(65.6))        9    25.7%  |||||
  5     (np.float64(47.1), np.float64(54.6))        8    22.9%  ||||
  6     (np.float64(6.4), np.float64(72.9))        5    14.3%  ||
  7     (np.float64(63.3), np.float64(19.9))        5    14.3%  ||
  8     (np.float

## ⚙️ 5. Construction du modèle MILP

Dans cette fonction, le but est de définir le modèle en utilisant la librairie Pulp dont vous trouverez la documentation ici : https://coin-or.github.io/pulp/.


In [ ]:
def build_cvrp_model(n, K, dist, demands, Q):
    # n : nombre total de noeuds (0 = dépôt, 1..n-1 = clients)
    # K : nombre de véhicules
    # dist : matrice des distances entre noeuds
    # demands : demande de chaque client
    # Q : capacité maximale d’un véhicule

    # Définitions des contraintes du problème (noeuds, clients, vehicules, prob, ...)
    noeuds    = list(range(n))
    clients  = list(range(1, n))
    vehicules = list(range(K))
    prob = pulp.LpProblem('CVRP_MTZ', pulp.LpMinimize)

    # Définitions des variables de décision x et u

    x = pulp.LpVariable.dicts(
        'x',
        [(i, j, k) for i in noeuds for j in noeuds for k in vehicules if i != j],
        cat='Binary'
    )

    u = pulp.LpVariable.dicts(
        'u',
        [(i, k) for i in clients for k in vehicules],
        lowBound=0,
        upBound=Q,
        cat='Continuous'
    )

    # Définition de la fonction objectif

    prob += pulp.lpSum(
        dist[i][j] * x[i, j, k]
        for i in noeuds
        for j in noeuds
        for k in vehicules
        if i != j
    )

    # Contrainte C1 : Unicité de la visite des noeuds

    for j in clients:
        prob += pulp.lpSum(x[i, j, k] for i in noeuds for k in vehicules if i != j) == 1


    # Contrainte C2 : Conservation du flux

    for i in clients:
        prob += pulp.lpSum()

    # C3 : Chaque véhicule part du dépôt au plus une fois

    # C4 : Chaque véhicule retourne au dépôt au plus une fois

    # C5-C6 : Contraintes MTZ (élimination des sous-tours + capacité)

    return


## 🚀 6. Résolution

In [ ]:
print('Construction du modele...')
prob, x, u = build_cvrp_model(n, N_vehicules, dist, demands, capacity)
print(f'  Variables   : {len(prob.variables())}')
print(f'  Contraintes : {len(prob.constraints)}')

print('\nResolution (CBC)...')
t0     = time.time()
status = prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=120))
elapsed = time.time() - t0

print(f'\nStatut   : {pulp.LpStatus[status]}')
print(f'Objectif : {pulp.value(prob.objective):.2f}')
print(f'Temps    : {elapsed:.2f}s')

## 📊 7. Extraction de la solution

In [ ]:
def extract_routes(x, n, K):
    # x : variables de décision du modèle (x[i,j,k])
    # n : nombre total de noeuds (0 = dépôt)
    # K : nombre de véhicules

    # Définitions des contraintes du problème (noeuds, clients, vehicules, routes, ...)
    noeuds = list(range(n))
    clients = list(range(1, n))
    vehicules = list(range(K))
    routes = [] # Liste qui va contenir toutes les tournées

    # Extraction itérative des routes de chaque vehicule:
    # Pour chaque vehicule,
    #   Sa route commence par le dépôt (0),
    #   puis en suivant les arcs, chaque client visité jusqu’au retour au dépôt est rajouté.
    # La route complète est ensuite ajoutée à la liste des tournées

    return routes

routes = extract_routes(x, n, N_vehicules)
total_dist = 0
print('Tournees optimales :')
for idx, route in enumerate(routes):
    load  = sum(demands[v] for v in route if v!=0)
    rdist = sum(dist[route[i]][route[i+1]] for i in range(len(route)-1))
    total_dist += rdist
    path = ' -> '.join(str(v) for v in route)
    print(f'  V{idx+1} | charge {int(load):2d}/{capacity} | dist {rdist:6.1f} | {path}')
print(f'\nDistance totale : {total_dist:.2f}')

## 📈 8. Visualisation des résultats

La fonction `plot_cvrp_solution` permet d'afficher les résulats du problème sur l'instance donnée.

La fonction `print_cvrp_summary` permet d'afficher un tableau récapitulatif des résultats.

In [ ]:
# @title
def plot_cvrp_solution(routes, coords, demands, dist, capacity, inst, status):
    """
    Visualise la solution d'un CVRP.

    Paramètres
    ----------
    routes   : list[list[int]]  — tournées extraites (incluant dépôt en 0 et n)
    coords   : np.ndarray (n, 2) — coordonnées de tous les nœuds (0 = dépôt)
    demands  : np.ndarray (n,)   — demandes (demands[0] = 0)
    dist     : np.ndarray (n, n) — matrice des distances
    capacity : int               — capacité d'un véhicule
    inst     : dict              — instance parsée (clés 'name', 'grid_size')
    status   : int               — code statut PuLP (pulp.LpStatus[status])
    """


    VEHICLE_COLORS        = ['#00d4ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b']
    BG, PANEL_BG          = '#0a0e1a', '#0f1421'
    TEXT_BRIGHT, TEXT_MID = '#f0f4ff', '#8892aa'
    grid_size             = inst.get('grid_size', 100)
    n                     = len(coords)
    depot_coord           = coords[0]

    total_dist = sum(
        dist[route[i]][route[i + 1]]
        for route in routes
        for i in range(len(route) - 1)
    )

    # ── Layout ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 11), facecolor=BG)
    gs  = fig.add_gridspec(
        2, 3, left=0.04, right=0.96, top=0.90, bottom=0.06,
        hspace=0.40, wspace=0.35
    )
    ax_map  = fig.add_subplot(gs[:, :2])
    ax_load = fig.add_subplot(gs[0, 2])
    ax_dist = fig.add_subplot(gs[1, 2])

    for ax in [ax_map, ax_load, ax_dist]:
        ax.set_facecolor(PANEL_BG)
        for spine in ax.spines.values():
            spine.set_edgecolor('#1e2a42')

    # ── Carte ─────────────────────────────────────────────────────────────
    ax_map.grid(color='#111827', linewidth=0.5, zorder=0)

    # Arcs des tournées
    for idx, route in enumerate(routes):
        color = VEHICLE_COLORS[idx % len(VEHICLE_COLORS)]
        for seg in range(len(route) - 1):
            i, j = route[seg], route[seg + 1]
            ax_map.plot(
                [coords[i, 0], coords[j, 0]],
                [coords[i, 1], coords[j, 1]],
                color=color, lw=5, alpha=0.08, zorder=2
            )
            ax_map.annotate(
                '', xy=(coords[j, 0], coords[j, 1]),
                xytext=(coords[i, 0], coords[i, 1]),
                arrowprops=dict(
                    arrowstyle='->', color=color, lw=1.8,
                    connectionstyle='arc3,rad=0.05', mutation_scale=15
                ),
                zorder=3
            )

    # Nœuds clients
    for i in range(1, n):
        vc = TEXT_MID
        for idx, route in enumerate(routes):
            if i in route:
                vc = VEHICLE_COLORS[idx % len(VEHICLE_COLORS)]
                break
        ax_map.scatter(*coords[i], s=300, color=vc, alpha=0.15, zorder=4)
        ax_map.scatter(*coords[i], s=160, color=PANEL_BG,         zorder=5)
        ax_map.scatter(*coords[i], s=130, color=vc,   alpha=0.9,  zorder=6)
        ax_map.text(
            coords[i, 0], coords[i, 1], str(i),
            fontsize=7, color=BG, ha='center', va='center',
            fontweight='bold', zorder=7
        )
        ax_map.text(
            coords[i, 0] + 1.5, coords[i, 1] + 2.5,
            f'd={int(demands[i])}',
            fontsize=5.5, color=TEXT_MID, zorder=7
        )

    # Dépôt
    ax_map.scatter(
        *depot_coord, s=520, color='white', marker='*',
        zorder=8, edgecolors='#ffd700', linewidths=1.5
    )
    ax_map.text(
        depot_coord[0], depot_coord[1] - 4, 'DÉPÔT',
        fontsize=7, color='white', ha='center', va='top',
        fontweight='bold', zorder=9
    )

    # Légende
    handles = []
    for idx, route in enumerate(routes):
        load  = sum(demands[v] for v in route if v != 0)
        rdist = sum(dist[route[i]][route[i + 1]] for i in range(len(route) - 1))
        handles.append(mpatches.Patch(
            facecolor=VEHICLE_COLORS[idx % len(VEHICLE_COLORS)],
            label=f'V{idx + 1}  charge {int(load)}/{capacity}  dist {rdist:.0f}',
            alpha=0.85
        ))
    handles.append(mpatches.Patch(facecolor='white', label='Dépôt', alpha=0.9))
    ax_map.legend(
        handles=handles, loc='lower left', framealpha=0.3,
        facecolor='#0a0e1a', edgecolor='#1e2a42',
        fontsize=8, labelcolor=TEXT_BRIGHT
    )
    ax_map.set_xlim(0, grid_size)
    ax_map.set_ylim(0, grid_size)
    ax_map.set_title(
        f'Solution optimale CVRP  |  Distance = {total_dist:.1f}',
        color=TEXT_BRIGHT, fontsize=13, fontweight='bold', pad=12
    )

    # ── Barres charge ──────────────────────────────────────────────────────
    lv   = [f'V{i + 1}' for i in range(len(routes))]
    lw   = [sum(demands[v] for v in r if v != 0) for r in routes]
    cv   = [VEHICLE_COLORS[i % len(VEHICLE_COLORS)] for i in range(len(routes))]
    bars = ax_load.bar(lv, lw, color=cv, alpha=0.85, width=0.55, zorder=3)
    ax_load.axhline(
        capacity, color='#ff6b6b', lw=1.5, ls='--',
        alpha=0.7, label=f'Cap={capacity}'
    )
    for b, val in zip(bars, lw):
        ax_load.text(
            b.get_x() + b.get_width() / 2, val + 0.5, str(int(val)),
            ha='center', va='bottom', color=TEXT_BRIGHT,
            fontsize=9, fontweight='bold'
        )
    ax_load.set_ylim(0, capacity * 1.2)
    ax_load.set_title('Charge par véhicule', color=TEXT_BRIGHT, fontsize=10, pad=8)
    ax_load.legend(fontsize=7, labelcolor=TEXT_BRIGHT, facecolor=BG, edgecolor='#1e2a42')
    ax_load.grid(axis='y', color='#111827', zorder=0)
    ax_load.tick_params(colors=TEXT_MID)

    # ── Barres distance ────────────────────────────────────────────────────
    dv    = [sum(dist[r[i]][r[i + 1]] for i in range(len(r) - 1)) for r in routes]
    bars2 = ax_dist.bar(lv, dv, color=cv, alpha=0.85, width=0.55, zorder=3)
    for b, val in zip(bars2, dv):
        ax_dist.text(
            b.get_x() + b.get_width() / 2, val + 0.5, f'{val:.0f}',
            ha='center', va='bottom', color=TEXT_BRIGHT,
            fontsize=9, fontweight='bold'
        )
    ax_dist.set_title('Distance par véhicule', color=TEXT_BRIGHT, fontsize=10, pad=8)
    ax_dist.grid(axis='y', color='#111827', zorder=0)
    ax_dist.tick_params(colors=TEXT_MID)

    # ── Titres globaux ─────────────────────────────────────────────────────
    fig.text(
        0.50, 0.95, 'CVRP  |  MILP (MTZ)  |  PuLP / CBC',
        ha='center', color=TEXT_BRIGHT, fontsize=15, fontweight='bold'
    )
    fig.text(
        0.50, 0.92,
        f'{inst["name"]}  |  statut : {pulp.LpStatus[status]}',
        ha='center', color=TEXT_MID, fontsize=9
    )

    plt.show()
    return fig

In [ ]:
# @title
def print_cvrp_summary(routes, demands, dist, capacity, N_vehicules, inst, prob, status):
    """
    Affiche le tableau récapitulatif d'une solution CVRP.

    Paramètres
    ----------
    routes     : list[list[int]]  — tournées extraites (dépôt en 0 et n)
    demands    : np.ndarray (n,)  — demandes (demands[0] = 0)
    dist       : np.ndarray (n,n) — matrice des distances
    capacity   : int              — capacité d'un véhicule
    N_vehicules : int              — nombre de véhicules
    inst       : dict             — instance parsée (clé 'name')
    prob       : pulp.LpProblem   — problème résolu
    status     : int              — code statut retourné par prob.solve()
    """
    import pulp

    total_dist = sum(
        dist[route[i]][route[i + 1]]
        for route in routes
        for i in range(len(route) - 1)
    )

    print('=' * 65)
    print(f'  {inst["name"]}')
    print('=' * 65)
    print(f'  {"Véhicule":<10}{"Tournée":<35}{"Charge":>7}{"Dist":>10}')
    print('-' * 65)

    for idx, route in enumerate(routes):
        load  = int(sum(demands[v] for v in route if v != 0))
        rdist = sum(dist[route[i]][route[i + 1]] for i in range(len(route) - 1))
        path  = '->'.join(str(v) for v in route)
        if len(path) > 33:
            path = path[:30] + '...'
        print(f'  V{idx + 1:<9} {path:<35} {load:>4}/{capacity} {rdist:>9.1f}')

    print('-' * 65)
    print(f'  {"TOTAL":<46} {int(demands[1:].sum()):>4}/{capacity * N_vehicules} {total_dist:>9.1f}')
    print('=' * 65)
    print(f'  Variables   : {len(prob.variables())}')
    print(f'  Contraintes : {len(prob.constraints)}')
    print(f'  Objectif    : {pulp.value(prob.objective):.4f}')
    print(f'  Statut      : {pulp.LpStatus[status]}')

In [ ]:
fig = plot_cvrp_solution(
    routes   = routes,
    coords   = coords,
    demands  = demands,
    dist     = dist,
    capacity = capacity,
    inst     = inst,
    status   = status,
)

In [ ]:
print_cvrp_summary(
    routes     = routes,
    demands    = demands,
    dist       = dist,
    capacity   = capacity,
    N_vehicules = N_vehicules,
    inst       = inst,
    prob       = prob,
    status     = status,
)

## 🗺️ 9. Reformulation à deux indices

Le but de cette partie est d'implémenter une version allégée du modèle CVRP classique.

Dans le modèle à trois indices, chaque variable $x_{ijk}$ précise quel véhicule $k$
emprunte quel arc — ce qui, pour des véhicules identiques, génère une **symétrie
inutile** : permuter les indices de véhicules produit des solutions différentes pour
le solveur mais rigoureusement équivalentes en pratique.

On propose ici une reformulation dans laquelle :

- Les variables de routage deviennent **$x_{ij} \in \{0,1\}$** : on indique simplement
  si l'arc $(i \to j)$ est emprunté, sans préciser par quel véhicule.
- La variable de charge **$u_i \geq 0$** perd également l'indice de véhicule.
- Le nombre de variables binaires passe de $O(Kn^2)$ à $O(n^2)$, soit un facteur
  $K$ de réduction.

Cette reformulation est valide sous l'hypothèse que les véhicules sont **identiques**
(même capacité, mêmes coûts), ce qui est le cas dans notre instance.

Après avoir proposé une modélisation de ce problème (cf. TD), implémenter une fonction
`build_cvrp_2idx_model(n, K, dist, demands, Q)` qui permet de construire le modèle *Pulp*.

In [ ]:
def build_cvrp_2idx_model(n, K, dist, demands, Q):
    # n : nombre total de noeuds (0 = dépôt, 1..n-1 = clients)
    # K : nombre de véhicules
    # dist : matrice des distances
    # demands : demande des clients
    # Q : capacité des véhicules

    # Définitions des ensembles et du problème
    noeuds   = list(range(n))
    clients  = list(range(1, n))
    prob = pulp.LpProblem('CVRP_2idx_MTZ', pulp.LpMinimize)

    # Variables de décision :

    # Fonction objectif : minimiser la distance totale

    # C1 : chaque client est visité exactement une fois

    # C2 : conservation du flux (entrées = sorties)

    # C3 : au plus K départs du dépôt (K véhicules max)

    # C4-C5 : contraintes MTZ (élimination des sous-tours + capacité)

    return

In [ ]:
print('Construction du modele...')
prob, x, u = build_cvrp_2idx_model(n, N_vehicules, dist, demands, capacity)
print(f'  Variables   : {len(prob.variables())}')
print(f'  Contraintes : {len(prob.constraints)}')

print('\nResolution (CBC)...')
t0     = time.time()
status = prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=120))
elapsed = time.time() - t0

print(f'\nStatut   : {pulp.LpStatus[status]}')
print(f'Objectif : {pulp.value(prob.objective):.2f}')
print(f'Temps    : {elapsed:.2f}s')

In [ ]:
def extract_routes_2idx(x, n, K):
    # x : variables de décision (x[i,j])
    # n : nombre total de noeuds (0 = dépôt)
    # K : nombre max de véhicules

    # Définitions des ensembles et structures
    noeuds, clients = list(range(n)), list(range(1, n))

    # Récupération des arcs utilisés (x ≈ 1)
    arcs = {(i,j) for i in noeuds for j in noeuds if i!=j
            if pulp.value(x[i,j]) is not None and pulp.value(x[i,j]) > 0.5}

    routes = []

    # Points de départ : arcs sortant du dépôt → débuts des tournées
    depot_exits = [j for (i,j) in arcs if i == 0]

    # Reconstruction des routes :
    # Pour chaque départ depuis le dépôt,
    #   on suit les arcs en ajoutant chaque client visité
    #   jusqu’au retour au dépôt
    # Chaque tournée est ensuite ajoutée à la liste des routes

    return routes


routes = extract_routes_2idx(x, n, N_vehicules)
total_dist = 0
print('Tournees optimales :')
for idx, route in enumerate(routes):
    load  = sum(demands[v] for v in route if v!=0)
    rdist = sum(dist[route[i]][route[i+1]] for i in range(len(route)-1))
    total_dist += rdist
    path = ' -> '.join(str(v) for v in route)
    print(f'  V{idx+1} | charge {int(load):2d}/{capacity} | dist {rdist:6.1f} | {path}')
print(f'\nDistance totale : {total_dist:.2f}')

In [ ]:
fig = plot_cvrp_solution(
    routes   = routes,
    coords   = coords,
    demands  = demands,
    dist     = dist,
    capacity = capacity,
    inst     = inst,
    status   = status,
)

## 🚐 10. Variante du problème 1

Le but de cette partie est d'implémenter une variante du problème de CVRP.

Dans le CVRP classique, l'objectif est de **minimiser la distance totale** parcourue
par l'ensemble de la flotte, avec un nombre de véhicules fixé à l'avance.

On considère ici une variante dans laquelle :

- Le **nombre de véhicules disponibles** $K$ est une borne supérieure (flotte maximale),
  et non plus un paramètre fixe.
- L'objectif devient **minimiser le nombre de véhicules effectivement utilisés**.
- En contrepartie, chaque véhicule ne peut pas parcourir une distance supérieure
  à un seuil $D_{\max}$ par tournée.

Cette variante modélise des situations réelles où le coût dominant est le
**coût fixe d'activation** d'un véhicule (chauffeur, amortissement, assurance)
plutôt que le coût kilométrique.


Après avoir proposé une modélisation de ce problème (cf. TD), implémenter une fonction `build_fleet_min_model(n, K, dist, demands, Q, D_max)` qui permet de construire le modèle *Pulp*.

In [ ]:
def build_fleet_min_model(n, K, dist, demands, Q, D_max):
    # n : nombre de noeuds (0 = dépôt)
    # K : nombre max de véhicules
    # dist : matrice des distances
    # demands : demande des clients
    # Q : capacité des véhicules
    # D_max : distance max par véhicule

    # Définitions des ensembles et du problème
    noeuds    = list(range(n))
    clients   = list(range(1, n))
    vehicules = list(range(K))
    prob = pulp.LpProblem('CVRP_FleetMin_MTZ', pulp.LpMinimize)

    # Variables de décision :

    # Objectif : minimiser le nombre de véhicules utilisés

    # C1 : chaque client est visité exactement une fois

    # C2 : conservation du flux (entrées = sorties)

    # C3' : départ du dépôt seulement si le véhicule est utilisé

    # C4' : retour au dépôt seulement si le véhicule est utilisé

    # C5-C6 : MTZ (élimination des sous-tours + capacité)

    # C8 : distance maximale par véhicule (active seulement si utilisé)

    return prob, x, u, y

In [ ]:
print('Construction du modele...')
prob, x, u, y = build_fleet_min_model(n, 5, dist, demands, capacity, D_max=100)
print(f'  Variables   : {len(prob.variables())}')
print(f'  Contraintes : {len(prob.constraints)}')

print('\nResolution (CBC)...')
t0     = time.time()
status = prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=360))
elapsed = time.time() - t0

print(f'\nStatut   : {pulp.LpStatus[status]}')
print(f'Objectif : {pulp.value(prob.objective):.2f}')
print(f'Temps    : {elapsed:.2f}s')

In [ ]:
routes = extract_routes(x, n, 5)
total_dist = 0
print('Tournees optimales :')
for idx, route in enumerate(routes):
    load  = sum(demands[v] for v in route if v!=0)
    rdist = sum(dist[route[i]][route[i+1]] for i in range(len(route)-1))
    total_dist += rdist
    path = ' -> '.join(str(v) for v in route)
    print(f'  V{idx+1} | charge {int(load):2d}/{capacity} | dist {rdist:6.1f} | {path}')
print(f'\nDistance totale : {total_dist:.2f}')

In [ ]:
fig = plot_cvrp_solution(
    routes   = routes,
    coords   = coords,
    demands  = demands,
    dist     = dist,
    capacity = capacity,
    inst     = inst,
    status   = status,
)

## ⚖️ 11. Variante du problème 2

Le but de cette partie est d'implémenter une seconde variante du problème de CVRP.

Dans le CVRP classique, l'objectif est de **minimiser la distance totale** parcourue
par l'ensemble de la flotte, sans se soucier de la répartition individuelle des tournées.

On considère ici une variante dans laquelle :

- Les **$K$ véhicules sont tous obligatoirement utilisés** : chaque employé effectue
  exactement une tournée complète.
- L'objectif devient **minimiser l'écart de distance** entre la tournée la plus longue
  et la tournée la plus courte, afin de répartir équitablement la charge de travail.

Cette variante modélise des situations réelles où le critère dominant est
l'**équité entre employés** (convention collective, temps de travail, fatigue),
plutôt que l'efficacité globale de la flotte.

Après avoir proposé une modélisation de ce problème (cf. TD), implémenter une fonction
`build_equity_model(n, K, dist, demands, Q)` qui permet de construire le modèle *Pulp*.

In [ ]:
def build_equity_model(n, K, dist, demands, Q):
    # n : nombre de noeuds (0 = dépôt)
    # K : nombre de véhicules
    # dist : matrice des distances
    # demands : demande des clients
    # Q : capacité des véhicules

    # Définitions des ensembles et du problème
    noeuds    = list(range(n))
    clients   = list(range(1, n))
    vehicules = list(range(K))
    prob = pulp.LpProblem('CVRP_Equity_MTZ', pulp.LpMinimize)

    # Variables de décision :

    # Objectif : minimiser l'écart entre les tournées (équité)

    # C1 : chaque client est visité exactement une fois

    # C2 : conservation du flux (entrées = sorties)

    # C3'' : chaque véhicule part du dépôt exactement une fois

    # C5-C6 : MTZ (élimination des sous-tours + capacité)

    # C10 : D_min <= distance de chaque tournée

    # C11 : D_max >= distance de chaque tournée

    return prob, x, u, D_min, D_max

In [ ]:
print('Construction du modele...')
prob, x, u, D_min, D_max = build_equity_model(n, N_vehicules, dist, demands, capacity)
print(f'  Variables   : {len(prob.variables())}')
print(f'  Contraintes : {len(prob.constraints)}')

print('\nResolution (CBC)...')
t0      = time.time()
status  = prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=360))
elapsed = time.time() - t0

print(f'\nStatut      : {pulp.LpStatus[status]}')
print(f'D_min       : {pulp.value(D_min):.2f}')
print(f'D_max       : {pulp.value(D_max):.2f}')
print(f'Ecart       : {pulp.value(D_max) - pulp.value(D_min):.2f}')
print(f'Temps       : {elapsed:.2f}s')

In [ ]:
routes = extract_routes(x, n, N_vehicules)
total_dist = 0
print('Tournees optimales :')
for idx, route in enumerate(routes):
    load  = sum(demands[v] for v in route if v!=0)
    rdist = sum(dist[route[i]][route[i+1]] for i in range(len(route)-1))
    total_dist += rdist
    path = ' -> '.join(str(v) for v in route)
    print(f'  V{idx+1} | charge {int(load):2d}/{capacity} | dist {rdist:6.1f} | {path}')
print(f'\nDistance totale : {total_dist:.2f}')

In [ ]:
fig = plot_cvrp_solution(
    routes   = routes,
    coords   = coords,
    demands  = demands,
    dist     = dist,
    capacity = capacity,
    inst     = inst,
    status   = status,
)